<a href="https://colab.research.google.com/github/shashidhar078/FlipGrid/blob/main/Hyperparameter_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install optuna
!pip install catboost
import pandas as pd
import optuna
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.2 MB/s eta 0:00:00


In [3]:
train = pd.read_csv("/content/processed_train_v1.csv")

y = train["demand"]
X = train.drop("demand", axis=1)

cat_features = X.select_dtypes(include=["object"]).columns.tolist()

In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [5]:
def objective(trial):

    params = {
        "iterations": trial.suggest_int("iterations", 800, 3000),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "random_strength": trial.suggest_float("random_strength", 0.1, 2),
        "loss_function": "RMSE",
        "verbose": 0,
        "random_state": 42
    }

    scores = []

    for train_idx, val_idx in kf.split(X):

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = CatBoostRegressor(**params)

        model.fit(
            X_train,
            y_train,
            cat_features=cat_features,
            verbose=False
        )

        preds = model.predict(X_val)
        score = r2_score(y_val, preds)
        scores.append(score)

    return sum(scores) / len(scores)

In [6]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

[I 2026-06-05 06:21:26,503] A new study created in memory with name: no-name-17b53932-2591-4c15-80cd-556d2ae0ea72
[I 2026-06-05 06:35:43,153] Trial 0 finished with value: 0.9555278339447959 and parameters: {'iterations': 1299, 'depth': 9, 'learning_rate': 0.03882074844621377, 'l2_leaf_reg': 2.741718547123665, 'random_strength': 0.16471046636655534}. Best is trial 0 with value: 0.9555278339447959.
[W 2026-06-05 06:46:03,628] Trial 1 failed with parameters: {'iterations': 2860, 'depth': 8, 'learning_rate': 0.07203959188250587, 'l2_leaf_reg': 3.1656367718674585, 'random_strength': 0.5299264550404786} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_1419/426109981.py", line 23, in objective
    model.fit(
  File "/usr/local/lib/python3.12/dist-packages/catboost/

KeyboardInterrupt: 

In [7]:
print("BEST SCORE:", study.best_value)
print("BEST PARAMS:", study.best_params)

BEST SCORE: 0.9555278339447959
BEST PARAMS: {'iterations': 1299, 'depth': 9, 'learning_rate': 0.03882074844621377, 'l2_leaf_reg': 2.741718547123665, 'random_strength': 0.16471046636655534}


In [8]:
best_params = study.best_params

best_model = CatBoostRegressor(
    **best_params,
    loss_function="RMSE",
    random_state=42,
    verbose=0
)

best_model.fit(X, y, cat_features=cat_features)

CatBoostRegressor(depth=9, iterations=1299, l2_leaf_reg=2.741718547123665, learning_rate=0.03882074844621377, loss_function='RMSE', random_state=42, random_strength=0.16471046636655534, verbose=0)